In [2]:
library(xgboost)
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [3]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [4]:
data=read.csv('Yearly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [5]:
dim(data)

[1] 23000    23

In [6]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9903832,1.535759e-07,5.174030,-0.94405361,0.47491778,0.6782989,0.3751645,⋯,0.28707837,-0.1820491,0.22316867,31,0.01651573,0.01651573,0.01651573,0,0,0
2,1,0,1,0.7455956,1.919834e-04,-2.797630,2.13419228,0.27486845,0.4580795,0.8010614,⋯,0.22214291,-0.3185895,0.37163112,31,0.01546788,0.01546788,0.01546788,0,0,0
3,1,0,1,0.9993711,1.038094e-09,5.497375,-0.01972653,0.20035534,0.4491332,0.3055037,⋯,0.37277176,-0.6280637,1.00110959,31,0.01541162,0.01541162,0.01541162,0,0,0
4,1,0,1,0.9991043,5.484677e-09,5.514672,-0.12400209,-0.18881555,0.3260450,0.3087948,⋯,0.60342537,-0.7758825,1.24511944,31,0.01328015,0.01328015,0.01328015,0,0,0
5,1,0,1,0.9941526,1.000137e-07,5.421624,0.72042474,0.15921568,0.4025341,0.3435267,⋯,0.16237458,-0.5492186,0.61790122,31,0.01211190,0.01211190,0.01211190,0,0,0
6,1,0,1,0.9642112,2.683180e-05,4.078327,0.73445937,0.02824047,0.1412002,0.5294464,⋯,0.08313085,0.1296658,0.09734006,19,0.01194334,0.01194334,0.01194334,0,0,0


In [7]:
dlist= load('Yearly_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [8]:
dim(MASE)

[1] 23000     5     4

In [9]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [10]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [11]:
nanum

NULL

In [12]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [13]:
table(realbestmin)

realbestmin
   0    1    2    3    4 
3429 2358 2940 4357 9916 

In [14]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [15]:
realbestmean

0
0
1
3
0
4
4
4
4
2
4


In [16]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9903832,1.535759e-07,5.174030,-0.94405361,0.47491778,0.6782989,0.3751645,⋯,0.28707837,-0.1820491,0.22316867,31,0.01651573,0.01651573,0.01651573,0,0,0
2,1,0,1,0.7455956,1.919834e-04,-2.797630,2.13419228,0.27486845,0.4580795,0.8010614,⋯,0.22214291,-0.3185895,0.37163112,31,0.01546788,0.01546788,0.01546788,0,0,0
3,1,0,1,0.9993711,1.038094e-09,5.497375,-0.01972653,0.20035534,0.4491332,0.3055037,⋯,0.37277176,-0.6280637,1.00110959,31,0.01541162,0.01541162,0.01541162,0,0,0
4,1,0,1,0.9991043,5.484677e-09,5.514672,-0.12400209,-0.18881555,0.3260450,0.3087948,⋯,0.60342537,-0.7758825,1.24511944,31,0.01328015,0.01328015,0.01328015,0,0,0
5,1,0,1,0.9941526,1.000137e-07,5.421624,0.72042474,0.15921568,0.4025341,0.3435267,⋯,0.16237458,-0.5492186,0.61790122,31,0.01211190,0.01211190,0.01211190,0,0,0
6,1,0,1,0.9642112,2.683180e-05,4.078327,0.73445937,0.02824047,0.1412002,0.5294464,⋯,0.08313085,0.1296658,0.09734006,19,0.01194334,0.01194334,0.01194334,0,0,0


In [17]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [18]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [19]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9903832,1.535759e-07,5.174030,-0.94405361,0.47491778,0.6782989,0.3751645,⋯,0.28707837,-0.1820491,0.22316867,31,0.01651573,0.01651573,0.01651573,0,0,0
2,1,0,1,0.7455956,1.919834e-04,-2.797630,2.13419228,0.27486845,0.4580795,0.8010614,⋯,0.22214291,-0.3185895,0.37163112,31,0.01546788,0.01546788,0.01546788,0,0,0
3,1,0,1,0.9993711,1.038094e-09,5.497375,-0.01972653,0.20035534,0.4491332,0.3055037,⋯,0.37277176,-0.6280637,1.00110959,31,0.01541162,0.01541162,0.01541162,0,0,0
4,1,0,1,0.9991043,5.484677e-09,5.514672,-0.12400209,-0.18881555,0.3260450,0.3087948,⋯,0.60342537,-0.7758825,1.24511944,31,0.01328015,0.01328015,0.01328015,0,0,0
5,1,0,1,0.9941526,1.000137e-07,5.421624,0.72042474,0.15921568,0.4025341,0.3435267,⋯,0.16237458,-0.5492186,0.61790122,31,0.01211190,0.01211190,0.01211190,0,0,0
6,1,0,1,0.9642112,2.683180e-05,4.078327,0.73445937,0.02824047,0.1412002,0.5294464,⋯,0.08313085,0.1296658,0.09734006,19,0.01194334,0.01194334,0.01194334,0,0,0


In [20]:
end_time = Sys.time()

In [21]:
time_matrix[1,]=end_time-start_time

In [22]:
end_time-start_time

Time difference of 10.12041 secs

## Target the interval where the actual error is minimum

In [23]:
start_time = Sys.time()

In [24]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [25]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [26]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:2.054349 
[2]	train-rmse:1.711905 
[3]	train-rmse:1.511460 
[4]	train-rmse:1.394788 
[5]	train-rmse:1.332264 
[6]	train-rmse:1.293268 
[7]	train-rmse:1.268623 
[8]	train-rmse:1.256257 
[9]	train-rmse:1.247049 
[10]	train-rmse:1.238409 
[11]	train-rmse:1.233591 
[12]	train-rmse:1.228891 
[13]	train-rmse:1.226931 
[14]	train-rmse:1.222875 
[15]	train-rmse:1.221130 
[16]	train-rmse:1.214927 
[17]	train-rmse:1.205064 
[18]	train-rmse:1.200693 
[19]	train-rmse:1.198064 
[20]	train-rmse:1.196013 
[21]	train-rmse:1.191804 
[22]	train-rmse:1.184007 
[23]	train-rmse:1.178860 
[24]	train-rmse:1.175367 
[25]	train-rmse:1.169202 
[26]	train-rmse:1.167569 
[27]	train-rmse:1.161780 
[28]	train-rmse:1.158237 
[29]	train-rmse:1.152403 
[30]	train-rmse:1.149482 
[31]	train-rmse:1.146164 
[32]	train-rmse:1.144756 
[33]	train-rmse:1.138348 
[34]	train-rmse:1.133116 
[35]	train-rmse:1.129023 
[36]	train-rmse:1.128036 
[37]	train-rmse:1.125749 
[38]	train-rmse:1.121791 
[39]	train-rmse:1.120

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [27]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-merror:0.507709 
[2]	train-merror:0.494943 
[3]	train-merror:0.485383 
[4]	train-merror:0.480942 
[5]	train-merror:0.475207 
[6]	train-merror:0.467374 
[7]	train-merror:0.461823 
[8]	train-merror:0.459109 
[9]	train-merror:0.448871 
[10]	train-merror:0.444492 
[11]	train-merror:0.439928 
[12]	train-merror:0.435365 
[13]	train-merror:0.431972 
[14]	train-merror:0.425867 
[15]	train-merror:0.419946 
[16]	train-merror:0.415443 
[17]	train-merror:0.410201 
[18]	train-merror:0.406377 
[19]	train-merror:0.404404 
[20]	train-merror:0.400086 
[21]	train-merror:0.395522 
[22]	train-merror:0.392439 
[23]	train-merror:0.387628 
[24]	train-merror:0.383681 
[25]	train-merror:0.381399 
[26]	train-merror:0.376650 
[27]	train-merror:0.373936 
[28]	train-merror:0.369187 
[29]	train-merror:0.362711 
[30]	train-merror:0.359812 
[31]	train-merror:0.356790 
[32]	train-merror:0.354200 
[33]	train-merror:0.347724 
[34]	train-merror:0.344702 
[35]	train-merror:0.341557 
[36]	train-merror:0.339583 
[

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [28]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011209 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4167
[LightGBM] [Info] Number of data points in the train set: 16214, number of used features: 17
[LightGBM] [Info] Start training from score 2.652399


In [29]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4167
[LightGBM] [Info] Number of data points in the train set: 16214, number of used features: 17
[LightGBM] [Info] Start training from score -1.916676
[LightGBM] [Info] Start training from score -2.267677
[LightGBM] [Info] Start training from score -2.047277
[LightGBM] [Info] Start training from score -1.658675
[LightGBM] [Info] Start training from score -0.844260


In [30]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [31]:
start_time = Sys.time()

In [32]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [33]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [34]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.920613 
[2]	train-rmse:1.644886 
[3]	train-rmse:1.484312 
[4]	train-rmse:1.393533 
[5]	train-rmse:1.342271 
[6]	train-rmse:1.312164 
[7]	train-rmse:1.293278 
[8]	train-rmse:1.282288 
[9]	train-rmse:1.271284 
[10]	train-rmse:1.264118 
[11]	train-rmse:1.258175 
[12]	train-rmse:1.253278 
[13]	train-rmse:1.243774 
[14]	train-rmse:1.239803 
[15]	train-rmse:1.236503 
[16]	train-rmse:1.230590 
[17]	train-rmse:1.223937 
[18]	train-rmse:1.220326 
[19]	train-rmse:1.219071 
[20]	train-rmse:1.210975 
[21]	train-rmse:1.205520 
[22]	train-rmse:1.202294 
[23]	train-rmse:1.195994 
[24]	train-rmse:1.194886 
[25]	train-rmse:1.192270 
[26]	train-rmse:1.185378 
[27]	train-rmse:1.179589 
[28]	train-rmse:1.178442 
[29]	train-rmse:1.171602 
[30]	train-rmse:1.165134 
[31]	train-rmse:1.162903 
[32]	train-rmse:1.158683 
[33]	train-rmse:1.154459 
[34]	train-rmse:1.152630 
[35]	train-rmse:1.146795 
[36]	train-rmse:1.139733 
[37]	train-rmse:1.137205 
[38]	train-rmse:1.133121 
[39]	train-rmse:1.127

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [35]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-merror:0.554644 
[2]	train-merror:0.531269 
[3]	train-merror:0.520970 
[4]	train-merror:0.514247 
[5]	train-merror:0.503577 
[6]	train-merror:0.498026 
[7]	train-merror:0.489392 
[8]	train-merror:0.480696 
[9]	train-merror:0.469471 
[10]	train-merror:0.463242 
[11]	train-merror:0.456642 
[12]	train-merror:0.449365 
[13]	train-merror:0.445171 
[14]	train-merror:0.437646 
[15]	train-merror:0.432096 
[16]	train-merror:0.425743 
[17]	train-merror:0.420624 
[18]	train-merror:0.414703 
[19]	train-merror:0.410694 
[20]	train-merror:0.407981 
[21]	train-merror:0.400456 
[22]	train-merror:0.395954 
[23]	train-merror:0.391452 
[24]	train-merror:0.383681 
[25]	train-merror:0.378500 
[26]	train-merror:0.375046 
[27]	train-merror:0.370544 
[28]	train-merror:0.368200 
[29]	train-merror:0.363205 
[30]	train-merror:0.358887 
[31]	train-merror:0.354015 
[32]	train-merror:0.347909 
[33]	train-merror:0.345196 
[34]	train-merror:0.342728 
[35]	train-merror:0.340385 
[36]	train-merror:0.338226 
[

Warning message in sprintf(save_name, env$iteration):
“one argument not used by format 'xgboost.model'”


In [36]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008785 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4167
[LightGBM] [Info] Number of data points in the train set: 16214, number of used features: 17
[LightGBM] [Info] Start training from score 2.345936


In [37]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008948 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4167
[LightGBM] [Info] Number of data points in the train set: 16214, number of used features: 17
[LightGBM] [Info] Start training from score -1.672703
[LightGBM] [Info] Start training from score -2.054951
[LightGBM] [Info] Start training from score -1.884901
[LightGBM] [Info] Start training from score -1.536547
[LightGBM] [Info] Start training from score -1.148238


In [38]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [39]:
end_time-start_time

Time difference of 2.409725 mins

## predict

In [40]:
start_time = Sys.time()

In [41]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [42]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [43]:
xgbregmin

[1]  2.337117e+00  1.792584e+00  1.255076e+00  2.551001e+00  2.596719e+00
    [6]  3.629522e+00  3.500438e+00  3.002187e+00  3.194767e+00  2.068563e+00
   [11]  3.329474e+00  2.666197e+00  3.643165e+00  2.764925e+00  2.296299e+00
   [16]  3.686271e+00  2.630901e+00  3.086411e+00  3.618511e+00  3.699216e+00
   [21]  3.726438e+00  3.014001e+00  1.547159e+00  6.633423e-01  3.779645e+00
   [26]  3.802778e+00  2.966493e+00  2.529742e+00  2.959569e+00  2.302070e+00
   [31]  9.342448e-01  3.201997e+00  2.745774e+00  2.815695e+00  3.594579e+00
   [36]  3.294819e+00  3.254361e+00  2.787058e+00  3.086627e+00  3.903009e+00
   [41]  3.219532e+00  1.747046e+00  1.406562e+00  1.663616e+00  1.826777e+00
   [46]  2.842401e+00  2.318129e+00  1.846675e+00  1.965235e+00  3.212933e+00
   [51]  3.346107e+00  2.787450e+00  4.191940e+00  4.083189e+00  3.504032e+00
   [56]  3.358683e+00  3.492638e+00  3.520531e+00  3.173833e+00  1.131840e+00
   [61]  3.714710e+00  3.267852e+00  3.914485e+00  3.178482e+00  3.006138e+00
   [66]  7.974321e-01  3.065506e+00  3.666938e+00  9.751269e-01  2.170453e+00
   [71]  2.993667e+00  3.030483e+00  1.891991e+00  2.900696e+00  3.792015e+00
   [76]  2.954782e+00  1.634407e+00  2.142831e+00  2.252886e+00  3.387010e+00
   [81]  3.079561e+00  4.018930e+00  2.461036e+00  2.314431e+00  1.144438e+00
   [86]  1.890475e+00  2.549591e+00  2.023514e+00  2.741017e+00  2.790500e+00
   [91]  1.195132e+00  2.371398e+00  1.863128e+00  3.121369e+00  1.665464e+00
   [96]  2.593374e+00  3.061486e+00  1.816560e+00  1.389577e+00  3.007826e+00
  [101]  1.970663e+00  2.542882e+00  1.903293e+00  2.384448e+00  1.230826e+00
  [106]  1.240252e+00  3.463725e-01  7.647082e-01  2.063627e+00  1.282611e+00
  [111]  1.941984e+00  2.044716e+00  3.887691e+00  7.309366e-01  1.663749e+00
  [116]  2.651313e+00  2.277994e+00  1.237767e+00  1.937222e+00  2.231965e+00
  [121]  2.118497e+00  1.861477e-01  2.782454e+00 -2.073485e-02  1.315456e+00
  [126]  3.771583e+00  2.040295e+00  2.225394e+00  2.797046e+00  2.783177e+00
  [131]  1.685492e+00  2.786710e+00  2.829902e+00  3.040083e+00  2.797736e+00
  [136]  3.832246e+00  3.304077e+00  3.102925e+00  2.420967e+00  3.475736e+00
  [141]  3.388039e+00  3.069445e+00  3.648266e+00  2.636831e+00  2.925277e+00
  [146]  3.025911e+00  3.165095e+00  3.253419e+00  3.695413e+00  2.547302e+00
  [151]  3.093494e+00  2.387420e+00  3.897276e+00  2.101495e+00  3.159014e+00
  [156]  2.147984e+00  3.489786e+00  2.519903e+00  2.762679e+00  2.662913e+00
  [161]  1.197287e+00  3.574236e+00  2.757144e+00  3.396778e+00  3.312926e+00
  [166]  3.489110e+00  3.265203e+00  1.649781e+00  1.749197e+00  1.779194e+00
  [171]  3.925676e+00  1.974084e+00  3.580540e+00  1.588349e+00  3.291673e+00
  [176]  1.377104e+00  2.016789e+00  1.795020e+00  1.948019e+00  3.724228e+00
  [181]  1.938262e+00  2.766421e+00  2.073862e+00  3.111390e+00  2.137681e+00
  [186]  2.360147e+00  9.150612e-01  3.722993e+00  3.560652e+00  1.858587e+00
  [191]  1.105842e+00  9.324335e-01  2.614937e+00  3.572067e+00  3.280902e+00
  [196]  2.863076e+00  9.331343e-01  2.512778e+00  1.497829e+00  3.274244e+00
  [201]  1.018695e+00  2.714927e+00  1.686126e+00  1.990806e+00  1.020031e+00
  [206]  1.993692e+00  1.653709e+00  3.467434e+00  1.108664e+00  3.974845e-01
  [211]  1.716845e+00  1.953380e+00  2.013638e+00  1.155925e+00  3.257348e+00
  [216]  2.782368e+00  6.131570e-01  9.080986e-01  2.178839e+00  3.298423e+00
  [221]  2.345844e+00  2.741657e+00  3.041255e+00  2.381347e+00  3.305056e+00
  [226]  1.952596e+00  2.715669e+00  1.655272e+00  3.361460e-01  2.011889e+00
  [231]  1.717308e+00  6.025215e-01  1.049529e+00  2.172374e+00  2.710884e-01
  [236]  3.284312e+00  2.074889e+00  1.401742e+00  1.069374e+00  4.030000e+00
  [241]  3.716665e+00  3.063015e+00  6.035996e-01  2.129334e+00  3.507771e+00
  [246]  3.802054e+00  3.012206e+00  3.089272e+00  2.263217e+00  3.387040e+00
  [251]  3.417131e+00  2.675101e+00  1.724598e+00  1.873122e+00  6.774582e-01
  [256]  2.338383e+00  3.5

In [44]:
xgbclsmin

[1] 3 0 1 3 4 4 4 4 4 2 4 4 4 4 1 4 4 4 4 4 4 4 2 4 4 4 4 4 4 4 1 4 4 4 4 4
   [37] 4 3 4 4 4 3 4 4 1 4 4 4 0 3 4 2 4 4 4 3 4 4 3 0 4 4 4 4 4 0 4 4 0 2 4 4
   [73] 1 4 4 4 4 4 4 4 4 4 4 4 1 2 4 4 3 2 3 2 4 4 2 3 4 2 1 4 2 3 4 4 0 0 4 1
  [109] 2 0 2 3 4 0 0 2 2 0 2 4 2 0 2 0 2 4 4 1 4 4 0 4 4 3 4 4 4 3 2 4 4 3 4 4
  [145] 4 2 4 4 4 2 4 4 4 3 1 3 4 4 4 3 2 4 2 4 4 4 4 2 4 4 4 2 3 3 4 0 4 2 1 4
  [181] 0 4 4 3 2 2 1 4 4 2 0 1 4 4 2 3 0 2 2 4 1 4 1 3 0 0 2 4 0 0 2 2 1 0 4 3
  [217] 0 4 2 4 4 4 4 0 4 2 4 1 0 2 1 1 0 4 0 4 2 2 0 4 4 3 0 3 4 4 3 3 0 4 4 4
  [253] 4 0 0 4 4 4 3 4 2 0 4 3 1 0 2 1 4 2 4 2 4 2 4 4 0 4 4 4 2 3 4 4 4 4 3 1
  [289] 1 4 2 4 4 1 0 4 4 4 0 4 4 4 2 4 1 2 1 0 2 2 1 0 1 1 1 0 4 2 4 2 4 4 4 4
  [325] 1 4 0 0 4 1 4 4 2 3 4 4 4 4 2 3 4 0 1 4 4 4 0 4 2 3 4 2 2 4 1 4 2 1 1 2
  [361] 4 3 4 3 3 0 4 4 4 4 4 3 4 2 2 0 4 3 4 3 4 4 4 4 4 4 4 2 4 4 3 4 4 4 4 1
  [397] 4 0 4 2 4 4 1 4 4 4 4 4 4 4 4 4 4 2 3 4 4 4 4 4 4 4 4 4 4 1 4 1 4 3 4 4
  [433] 3 4 1 1 1 3 3 4 2 4 4 4 4 4 4 4 1 4 4 4 4 1 4 3 4 4 0 4 2 1 4 1 4 4 3 4
  [469] 4 0 1 0 2 2 0 4 4 3 4 4 3 1 1 4 3 1 2 4 0 1 4 4 4 4 4 4 4 4 0 3 4 1 1 4
  [505] 4 4 4 4 4 3 4 3 0 4 4 4 2 4 2 4 4 3 4 4 0 1 4 3 1 4 4 4 3 4 3 4 3 1 4 4
  [541] 4 1 0 4 2 4 2 4 4 1 4 4 4 4 4 4 4 4 2 0 4 4 4 4 4 4 4 4 4 4 4 4 4 3 4 4
  [577] 4 4 4 4 1 3 0 2 0 4 0 4 4 1 4 4 3 4 4 4 4 0 0 2 3 0 0 4 4 1 4 4 4 2 1 4
  [613] 2 2 0 4 4 4 4 0 4 4 4 4 4 4 1 0 2 2 0 2 4 4 4 4 4 2 4 0 4 4 4 4 4 4 2 4
  [649] 0 4 3 4 4 1 4 0 1 4 4 0 4 0 1 0 0 4 0 4 2 3 1 4 3 4 3 4 4 4 4 3 4 4 4 4
  [685] 4 1 4 4 4 3 2 0 4 1 4 2 4 0 0 1 4 4 4 1 1 0 0 4 4 4 4 1 1 4 3 0 4 0 0 4
  [721] 0 3 4 4 4 4 2 4 2 0 4 2 1 4 4 0 0 4 1 3 0 3 0 2 1 1 1 1 4 1 4 4 4 4 3 4
  [757] 4 4 4 4 0 4 4 3 2 3 3 4 1 0 4 4 0 4 4 4 1 2 4 4 4 4 1 4 4 0 0 4 4 4 4 0
  [793] 1 3 0 2 0 3 2 4 1 4 1 0 1 0 4 2 4 1 4 4 4 4 0 4 4 3 4 3 4 4 0 2 3 2 4 0
  [829] 1 1 4 0 4 1 4 4 0 4 0 0 0 3 1 0 4 4 3 0 1 4 2 3 4 4 3 4 1 4 0 0 1 4 4 0
  [865] 4 2 4 1 4 4 4 4 2 4 2 4 4 4 4 4 3 1 4 4 4 0 4 4 4 3 4 2 3 4 0 0 2 3 1 4
  [901] 4 0 4 4 4 1 4 4 4 2 0 4 3 4 4 4 2 4 4 4 4 2 4 0 4 0 0 4 3 1 4 4 4 4 1 3
  [937] 4 4 3 3 4 2 4 4 4 3 4 0 4 3 4 1 3 1 1 2 2 4 4 4 1 1 4 0 4 2 3 4 4 2 0 0
  [973] 4 4 4 4 0 4 2 4 3 1 3 0 4 4 4 4 0 4 2 4 4 4 4 4 0 4 4 4 4 4 4 3 4 4 4 4
 [1009] 4 4 4 3 4 4 4 3 4 0 3 1 4 4 2 4 4 4 4 3 4 4 0 4 4 4 2 2 0 3 4 4 4 4 0 1
 [1045] 4 4 4 4 1 3 0 4 0 2 4 0 1 0 2 4 4 4 3 4 4 4 3 4 4 2 3 4 1 3 0 4 1 0 4 0
 [1081] 4 3 3 1 4 1 3 2 4 0 3 1 4 4 3 3 3 4 0 4 2 1 3 3 1 4 4 3 1 4 4 4 4 0 4 0
 [1117] 3 2 0 2 3 4 4 4 4 4 3 0 3 1 4 4 4 0 2 3 4 1 0 0 4 4 4 1 4 3 3 3 4 4 3 4
 [1153] 2 4 4 4 4 2 3 4 1 4 1 4 2 3 3 0 1 0 3 4 2 4 4 4 4 4 4 2 2 4 4 0 1 2 3 4
 [1189] 4 2 4 4 0 1 4 0 4 2 2 4 4 4 4 4 3 3 4 3 4 1 1 4 4 2 3 4 3 4 4 1 0 4 4 4
 [1225] 1 4 2 4 4 4 4 4 4 3 4 3 1 3 1 3 4 4 3 0 3 4 4 2 4 4 1 4 4 0 0 4 2 2 4 3
 [1261] 4 0 2 0 0 0 4 4 4 4 4 4 2 2 4 4 4 4 4 4 4 3 3 4 4 4 3 0 1 4 4 4 1 4 4 4
 [1297] 4 4 2 4 4 2 4 4 4 4 4 3 4 0 4 4 4 4 4 3 4 4 0 4 1 4 3 4 3 4 4 4 3 3 0 3
 [1333] 4 4 1 4 4 4 3 4 4 4 4 3 4 4 4 4 3 4 1 4 3 4 0 3 4 3 4 4 4 4 4 3 3 3 3 0
 [1369] 3 4 4 2 4 4 3 3 1 4 3 4 4 4 3 3 4 0 4 4 3 4 4 4 1 0 4 4 4 1 4 4 4 0 4 4
 [1405] 4 4 1 4 4 0 4 4 4 4 4 4 1 2 4 4 0 4 4 2 4 4 4 4 0 2 4 1 0 4 2 4 3 4 3 1
 [1441] 3 1 4 3 4 4 4 2 4 4 4 4 4 3 3 4 4 4 0 4 4 4 4 4 4 4 4 4 4 2 4 4 1 4 4 4
 [1477] 4 2 4 4 2 4 4 4 4 4 0 4 4 2 4 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 4 4
 [1513] 4 4 4 4 3 4 2 4 4 4 4 3 3 4 4 4 4 4 0 1 4 4 0 4 4 4 4 4 4 0 4 3 4 4 0 4
 [1549] 4 4 4 4 4 4 2 4 3 3 4 4 4 4 1 2 4 4 4 4 4 4 4 4 4 4 3 4 4 4 2 3 4 4 4 4
 [1585] 1 4 4 4 4 4 4 4 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 4 4 4 4 3 3 4 4 4 3
 [1621] 4 4 4 2 2 3 4 4 4 4 2 2 2 2 4 4 4 4 4 1 0 1 4 4 4 0 2 0 4 0 4 2 2 3 4 4
 [1657] 1 4 4 4 4 4 4 4 4 0 0 2 4 0 3 4 4 4 4 4 0 4 4 4 4 2 4 1 0 4 4 3 3 4 4 2
 [1693] 4 4 4 4 4 3 0 3 3 2 1 4 4 4 4 3 4 3 4 4 1 4 4 4 3 3 4 4 2 4 4 4 4 4 4 4
 [1729] 2 4 4 4 4 4 4 4 4 1 1 1 1 2 4 4 2 2 4 4 4 0 4 1 4 4 4 2 4 3 4 4 3 2 3 4
 [1765] 4 2 2 4 4 2 2 2 4 2 4 4 4 4 4 4 0 4 4 4 4 4 4 4 1 2 4 0 3 3 4 3 4 4 4 4
 [18

In [45]:
lgbregmin

[1] 2.0940595 1.9779081 1.5594698 2.4796076 2.4899747 3.5399693 3.3242246
    [8] 2.6342503 3.1362022 2.3389487 3.1727935 3.1004212 3.5412325 1.9371410
   [15] 2.5606435 3.7123772 2.3518069 2.9517368 3.7133360 3.3710457 3.3187945
   [22] 3.2659890 1.5227444 0.9820881 3.3788747 3.2160074 3.1312969 3.4008380
   [29] 3.3204317 2.3367371 1.3329351 3.2092280 2.9540785 3.2592203 3.4371401
   [36] 3.3917249 3.3837020 3.0576098 3.4268718 3.5125774 3.6076439 2.4200612
   [43] 2.7420555 2.6729015 2.1742214 2.9078252 2.7851128 2.5046209 2.4368083
   [50] 2.9523967 2.9874834 3.0953358 4.0651579 3.3653389 3.1203165 3.5848219
   [57] 3.2840717 2.9307718 3.2932973 1.3561421 3.4729559 2.6735770 3.4847676
   [64] 2.6623221 2.8015423 1.9338655 3.0428946 3.2987712 1.1472544 2.0590516
   [71] 2.6611292 2.8480096 1.4791450 3.2841219 3.2558043 2.9961081 2.5193103
   [78] 2.9150473 3.3681465 3.6426015 3.1514662 3.6468864 3.3122183 2.1326338
   [85] 2.1683849 2.3851594 2.6381347 2.5571194 2.8188910 2.5163185 2.3844720
   [92] 2.3167313 2.1165191 2.8798532 1.7152165 2.4644542 2.1772166 1.6843476
   [99] 1.6006858 2.8278911 1.8396769 2.7566325 2.0682096 2.0852323 1.6167453
  [106] 1.9859638 1.6522854 1.3082557 2.4739149 2.0437541 2.4023379 1.5636377
  [113] 2.9574477 1.9037848 1.5337991 2.6771970 1.9281762 1.3487445 2.0776899
  [120] 2.4513323 2.1671474 1.0735235 2.6663658 1.2210907 1.5939357 2.4294550
  [127] 2.7461317 2.8577666 2.0477647 2.6845889 2.3742809 2.4974325 2.9157602
  [134] 3.3143452 2.7833060 3.3619884 3.0174957 2.8302664 2.6942198 3.4199886
  [141] 3.4961131 3.1885472 3.3385128 3.1409742 2.8154046 2.5069257 3.1649218
  [148] 3.0027439 3.1829882 2.5303644 2.8355470 3.4306223 2.9903963 2.0736013
  [155] 3.2388331 3.0986166 3.1605310 3.0686403 2.3611701 3.0703736 2.3896112
  [162] 3.4943451 3.2890951 3.0086095 3.4768555 3.0369620 2.9702069 2.2700360
  [169] 2.1902311 3.0381627 3.4777118 1.9627914 3.4051620 2.7447369 3.1686060
  [176] 1.1054048 3.1081102 2.0258252 2.7721812 3.2632660 2.3613917 2.9305753
  [183] 3.0595791 2.4585989 2.1309083 2.1881745 2.1642189 2.3329662 2.5359619
  [190] 2.2330057 1.2159994 1.4309694 2.3149476 3.3753747 3.1470246 2.6431306
  [197] 1.6917326 2.4074160 1.6518761 2.4312044 1.6991146 1.9758222 1.8334380
  [204] 1.9962000 1.6395483 1.8816822 1.5395854 3.4373171 1.9656119 1.7335992
  [211] 1.8024130 1.8155500 2.1044018 1.5265164 2.4170142 2.4288374 2.0632218
  [218] 2.2709959 2.6519527 3.1522462 2.9678180 3.0081028 2.3487223 1.8316609
  [225] 2.8494635 2.0654089 2.5255766 1.9758443 1.1965622 2.0740844 1.5297039
  [232] 1.2297390 1.8352907 2.3067531 2.0924917 2.6705568 2.6543748 1.5355691
  [239] 2.5449240 3.5380468 3.7051488 3.2169130 1.3876431 3.4588617 3.4428312
  [246] 3.4828788 3.5355857 3.1559201 2.0201425 3.2516059 3.3339492 3.3166233
  [253] 2.6191952 2.8667111 0.8632401 2.5590516 2.8120692 2.9966521 2.3506475
  [260] 3.0591853 2.3645514 1.5289661 3.0859671 2.6334617 2.6950548 0.9098828
  [267] 2.6530490 2.1102500 3.1274746 1.9496420 1.8960421 1.9993984 3.3806761
  [274] 2.3860758 3.4711219 3.1174485 1.2411146 2.8136522 3.1344118 2.4769353
  [281] 2.2617363 2.2663235 3.3144583 1.9506372 2.9426903 2.2602982 1.9172145
  [288] 2.7069549 2.1737069 2.2661249 2.3170739 3.5055136 3.7789457 1.9269461
  [295] 1.0743571 2.7440195 2.3996853 3.2829021 1.7313292 3.4710587 3.1196649
  [302] 3.2612102 2.2491854 2.6304589 1.4603118 1.7521018 2.2731717 2.5332442
  [309] 1.3017693 1.8585739 2.0097877 1.4201390 1.6482368 1.6260111 1.8142612
  [316] 1.8668869 2.3312665 1.4243153 2.2812202 1.9383815 3.2807771 2.7908353
  [323] 3.0887569 1.9515848 1.6206782 2.1499664 1.6383211 2.3858575 3.3351065
  [330] 1.3898530 3.3634296 1.9083181 2.1768271 3.0992969 2.2170299 2.5555021
  [337] 2.5241709 2.9633109 3.1428352 1.9142970 2.2473307 1.5428013 1.9956715
  [344] 2.0420314 3.2300339 2.5194608 1.5038721 3.1088844 2.3290728 3.1472076
  [351] 2.3073497 1.5520962 1.8169431 3.4488060 2.4831723 2.3261621 1.6521554
  [358] 1.8174196 1.551307

In [46]:
lgbclsmin

0.11820129,0.11073954,0.12212549,0.40273598,0.2461977
0.39125011,0.06710329,0.07239889,0.11725205,0.3519957
0.10023454,0.43481672,0.16374003,0.17030878,0.1308999
0.08339521,0.12985937,0.16171648,0.43100016,0.1940288
0.07780959,0.18311479,0.08733764,0.29398670,0.3577513
0.05651462,0.06881728,0.02057611,0.04011412,0.8139779
0.05893760,0.10527460,0.01260033,0.07721942,0.7459681
0.16010521,0.05311854,0.06826877,0.09769072,0.6208168
0.06156876,0.04802166,0.07550915,0.12387367,0.6910268
0.15719416,0.05137750,0.36746719,0.09079477,0.3331664
0.09607358,0.03233565,0.04057739,0.05989839,0.7711150


In [47]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [48]:
lgbclsminr

1
2
3
2
3
4
0
2
1
3
4


In [49]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [50]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [51]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [52]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
3,2.337117,1,2.094060
0,1.792584,2,1.977908
1,1.255076,3,1.559470
3,2.551001,2,2.479608
4,2.596719,3,2.489975
4,3.629522,4,3.539969


In [53]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [54]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
0,1.123992,1,1.339693
0,1.363831,2,1.414877
1,1.152056,3,1.256118
3,2.121927,1,2.138096
0,1.961321,2,1.753921
4,3.597152,4,3.627251


In [55]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
0,1.123992,1,1.339693
0,1.363831,2,1.414877
1,1.152056,3,1.256118
3,2.121927,1,2.138096
0,1.961321,2,1.753921
4,3.597152,4,3.627251


In [56]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [57]:
result_list=list(preallmin,preallmean,time_matrix)

In [58]:
save(result_list, file = "Yearly_nnetar_opt_pre_result.RData")

In [59]:
time_matrix

user_time,system_time,elapsed_time
10.120406,10.120406,10.120406
2.415596,2.415596,2.415596
2.409725,2.409725,2.409725
1.677931,1.677931,1.677931
